# DefectosCafeVerde — train-only expert routability audit
Menilai apakah komplementaritas D0/AF2/DCWCF/LIF/RAFC dapat dirutekan dari fitur yang tersedia saat inferensi. Tidak melatih detector, tidak memakai validation, dan tidak mengekstrak test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import importlib, json, os, shutil, subprocess, sys, tarfile
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'
BRANCH='codex/defectoscafeverde-error-complementarity'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True,cwd=WORK)
import torch
print('TORCH PRE-INSTALL:',torch.__version__,'| CUDA BUILD:',torch.version.cuda,'| CUDA AVAILABLE:',torch.cuda.is_available())
if not torch.cuda.is_available():
    subprocess.run(['nvidia-smi'],check=False)
    raise RuntimeError('PyTorch GPU tidak tersedia SEBELUM instalasi. Restart runtime, pastikan T4 terhubung, lalu jalankan sel ini dari awal.')
torch_version_before=torch.__version__
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','ultralytics==8.4.96','ultralytics-thop'],check=True,cwd=WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True,cwd=WORK)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
if torch.__version__!=torch_version_before or not torch.cuda.is_available(): raise RuntimeError('Instalasi mengubah/merusak PyTorch CUDA; restart runtime dan laporkan TORCH PRE-INSTALL.')
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
from coffee_detector.experiments.prepare_defectoscafeverde_routability_colab import prepare_inputs
PROJECT,ARCHIVE,RESULTS,CHECKPOINTS,SOURCES=prepare_inputs(WORK,'/content/drive')
OUT=PROJECT/'experiments/defectoscafeverde-train-routability-v1'; OUT.mkdir(parents=True,exist_ok=True)
print('PROJECT:',PROJECT)
print('ARCHIVE:',ARCHIVE)
print('SOURCES:',json.dumps(SOURCES,indent=2))

In [ ]:
import yaml
DATA=WORK/'defectoscafeverde-train-routability'
if DATA.exists(): shutil.rmtree(DATA)
DATA.mkdir(parents=True)
prefix='grouped-physical-v1/'; allowed={'data.yaml','grouped_audit.json','grouped_summary.json'}
with tarfile.open(ARCHIVE,'r') as bundle:
    for member in bundle.getmembers():
        name=member.name.replace('\\','/')
        if not name.startswith(prefix): continue
        relative=name[len(prefix):].strip('/')
        if not relative: continue
        first=relative.split('/',1)[0]
        if not (first=='train' or relative in allowed): continue
        target=(DATA/relative).resolve()
        if DATA.resolve() not in target.parents and target!=DATA.resolve(): raise RuntimeError('Unsafe archive path')
        if member.isdir(): target.mkdir(parents=True,exist_ok=True); continue
        target.parent.mkdir(parents=True,exist_ok=True); source=bundle.extractfile(member)
        if source is None: raise RuntimeError(f'Gagal membaca {member.name}')
        with source,target.open('wb') as output: shutil.copyfileobj(source,output)
if (DATA/'val').exists() or (DATA/'test').exists(): raise RuntimeError('VAL/TEST TEREXPOSE - STOP')
payload=yaml.safe_load((DATA/'data.yaml').read_text()); payload.pop('val',None); payload.pop('test',None)
payload['path']=str(DATA); payload['train']='train/images'
(DATA/'data.yaml').write_text(yaml.safe_dump(payload,sort_keys=False),encoding='utf-8')
print('TRAIN:',len(list((DATA/'train/images').glob('*'))),'VAL:',(DATA/'val').exists(),'TEST:',(DATA/'test').exists())

In [ ]:
from coffee_detector.analysis.defectoscafeverde_train_routability import run_train_routability_audit
SUMMARY=OUT/'defectoscafeverde_train_routability.json'
RECORDS=OUT/'defectoscafeverde_train_routability_targets.jsonl'
result=run_train_routability_audit(DATA,RESULTS,CHECKPOINTS,SUMMARY,records_output=RECORDS,device='0')
print('SUMMARY:',SUMMARY)
print('RECORDS:',RECORDS)

In [ ]:
import pandas as pd
selection=result['minimal_subset']; router=result['oof_router']
display(pd.DataFrame([{
 'subset':' + '.join(selection['subset']),
 'full_oracle_gain':selection['full_oracle_gain'],
 'subset_oracle_gain':selection['subset_oracle_gain'],
 'retained_gain':selection['retained_full_gain_fraction'],
 'd0_recall':router['d0_recall'],
 'best_standalone_recall':router['best_standalone_recall'],
 'max_confidence_recall':router['max_confidence_recall'],
 'oof_router_recall':router['router_recall'],
 'subset_oracle_recall':router['subset_oracle_recall'],
 'router_gain_over_d0':router['router_gain_over_d0'],
 'captured_oracle_gain':router['captured_oracle_gain_fraction']}]).style.format({
 key:'{:.2%}' for key in ('full_oracle_gain','subset_oracle_gain','retained_gain','d0_recall','best_standalone_recall','max_confidence_recall','oof_router_recall','subset_oracle_recall','router_gain_over_d0','captured_oracle_gain')}))
print('FOLD SUBSETS:',router['fold_subsets'])
print('FULL-TRAIN LABEL COUNTS:',router['full_train_subset_label_counts'])
print('SELECTION COUNTS:',router['selection_counts'])
print('CRITERIA:',result['criteria'])
print('DECISION:',result['decision'])
print('NEXT:',result['next'])
print('DETECTOR TRAINING:',result['detector_training_executed'],'| VALIDATION:',result['validation_images_accessed'],'| TEST:',result['test_images_accessed'])
print('Kirim tabel, FOLD SUBSETS, FULL-TRAIN LABEL COUNTS, SELECTION COUNTS, CRITERIA, DECISION, dan NEXT. Jangan menjalankan training atau membuka validation/test.')